# 04a — WITH imbalance handling: 3 CNNs (5 seeds)

Identical to 03a **except** the loss is a class-weighted CrossEntropyLoss (single-layer inverse-frequency handling). This is the 'with handling' arm of the controlled comparison; 03a is the 'without' arm. Only the loss weighting differs. Files are saved with a `_bal` suffix so both arms coexist. Combine both arms in 04c.

In [1]:
# CONFIG
DATA_ROOT = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"
OUT_DIR   = "/kaggle/working"
SEEDS     = [42, 123, 2025, 7, 99]
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5
NUM_WORKERS = 2

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f"Missing path: {pth}"
print("Paths OK | seeds:", SEEDS, "| workers:", NUM_WORKERS)

Paths OK | seeds: [42, 123, 2025, 7, 99] | workers: 2


In [2]:
!pip install timm --quiet
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score,
    precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_idx = np.load(f"{SPLIT_DIR}/clean_train_indices.npy").tolist()
test_idx  = np.load(f"{SPLIT_DIR}/clean_test_indices.npy").tolist()
class_names = open(f"{SPLIT_DIR}/class_names.txt").read().splitlines()
SEVERE_IDX = class_names.index("Severe")
print("Classes:", class_names)

# --- inverse-frequency class weights from the TRAINING split (single-layer handling) ---
base = datasets.ImageFolder(DATA_ROOT)
all_labels = [base.samples[i][1] for i in range(len(base.samples))]
train_labels = [all_labels[i] for i in train_idx]
counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(float)
inv = 1.0 / counts
class_weights = torch.tensor(inv / inv.sum(), dtype=torch.float).to(device)
print("Class weights (inverse freq, normalised):",
      {class_names[i]: round(float(class_weights[i]),4) for i in range(NUM_CLASSES)})

Device: cuda
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Class weights (inverse freq, normalised): {'Mild': 0.2334, 'Moderate': 0.312, 'No_DR': 0.1272, 'Proliferate_DR': 0.1132, 'Severe': 0.2142}


In [3]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator(); g.manual_seed(seed)
    return g

In [4]:
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)), transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20), transforms.ColorJitter(0.3,0.3,0.2,0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, seed, crop_from=None):
    g = set_seed(seed)
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True, num_workers=NUM_WORKERS, generator=g),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=NUM_WORKERS))

In [5]:
def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        run = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,"logits") else out, y)
            loss.backward(); optimizer.step()
            run += loss.item()
        if scheduler: scheduler.step()
        print(f"    epoch {ep+1}/{epochs} loss {run/len(loader):.4f}")
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        out = model(x.to(device))
        out = out.logits if hasattr(out,"logits") else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def save_and_report(name, seed, model, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")
    pr,rc,f1,sup = precision_recall_fscore_support(labels,preds,labels=range(NUM_CLASSES),zero_division=0)
    torch.save(model.state_dict(), f"{OUT_DIR}/{name}_bal_seed{seed}.pth")
    np.savez(f"{OUT_DIR}/{name}_bal_seed{seed}_preds.npz", probs=probs, preds=preds, labels=labels)
    print(f"  [{name}+bal seed {seed}] acc {acc:.4f} | macroF1 {f1m:.4f} | "
          f"SevereRec {rc[SEVERE_IDX]:.3f} | ProlifRec {rc[class_names.index('Proliferate_DR')]:.3f}")
    print(f"  saved {name}_bal_seed{seed}.pth + _preds.npz")

def already_done(name, seed):
    p = f"{OUT_DIR}/{name}_bal_seed{seed}_preds.npz"
    if os.path.exists(p):
        print(f"  [skip] {name}+bal seed {seed} already done"); return True
    return False

### ConvNeXt-Tiny × 5 (weighted loss)

In [6]:
for s in SEEDS:
    if already_done("convnext_tiny", s): continue
    print(f"ConvNeXt-Tiny+bal seed {s}"); tr, te = make_loaders(224, s)
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES); m = m.to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    opt = optim.Adam(m.parameters(), lr=1e-4)
    m = train_model(m, tr, crit, opt)
    probs, labels = evaluate(m, te); save_and_report("convnext_tiny", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

ConvNeXt-Tiny+bal seed 42
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 240MB/s] 


    epoch 1/10 loss 0.2654
    epoch 2/10 loss 0.1040
    epoch 3/10 loss 0.0781
    epoch 4/10 loss 0.0654
    epoch 5/10 loss 0.0790
    epoch 6/10 loss 0.0552
    epoch 7/10 loss 0.0358
    epoch 8/10 loss 0.0343
    epoch 9/10 loss 0.0441
    epoch 10/10 loss 0.0298
  [convnext_tiny+bal seed 42] acc 0.8657 | macroF1 0.8493 | SevereRec 0.667 | ProlifRec 0.787
  saved convnext_tiny_bal_seed42.pth + _preds.npz
ConvNeXt-Tiny+bal seed 123
    epoch 1/10 loss 0.2817
    epoch 2/10 loss 0.0877
    epoch 3/10 loss 0.0629
    epoch 4/10 loss 0.0711
    epoch 5/10 loss 0.0669
    epoch 6/10 loss 0.0538
    epoch 7/10 loss 0.0557
    epoch 8/10 loss 0.0249
    epoch 9/10 loss 0.0501
    epoch 10/10 loss 0.0405
  [convnext_tiny+bal seed 123] acc 0.9310 | macroF1 0.9137 | SevereRec 0.736 | ProlifRec 0.921
  saved convnext_tiny_bal_seed123.pth + _preds.npz
ConvNeXt-Tiny+bal seed 2025
    epoch 1/10 loss 0.2995
    epoch 2/10 loss 0.1068
    epoch 3/10 loss 0.0643
    epoch 4/10 loss 0.0744
    e

### Inception V3 × 5 (weighted loss)

In [7]:
for s in SEEDS:
    if already_done("inception_v3", s): continue
    print(f"Inception V3+bal seed {s}"); tr, te = make_loaders(299, s, crop_from=320)
    m = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    m.AuxLogits.fc = nn.Linear(m.AuxLogits.fc.in_features, NUM_CLASSES); m = m.to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    opt = optim.Adam(m.parameters(), lr=1e-4)
    m = train_model(m, tr, crit, opt, aux=True)
    probs, labels = evaluate(m, te); save_and_report("inception_v3", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

Inception V3+bal seed 42
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 183MB/s] 


    epoch 1/10 loss 0.4447
    epoch 2/10 loss 0.1732
    epoch 3/10 loss 0.1758
    epoch 4/10 loss 0.1344
    epoch 5/10 loss 0.1233
    epoch 6/10 loss 0.0930
    epoch 7/10 loss 0.0907
    epoch 8/10 loss 0.0569
    epoch 9/10 loss 0.0624
    epoch 10/10 loss 0.0624
  [inception_v3+bal seed 42] acc 0.8694 | macroF1 0.8796 | SevereRec 0.954 | ProlifRec 0.616
  saved inception_v3_bal_seed42.pth + _preds.npz
Inception V3+bal seed 123
    epoch 1/10 loss 0.4561
    epoch 2/10 loss 0.1876
    epoch 3/10 loss 0.1627
    epoch 4/10 loss 0.1147
    epoch 5/10 loss 0.0940
    epoch 6/10 loss 0.0908
    epoch 7/10 loss 0.0730
    epoch 8/10 loss 0.0701
    epoch 9/10 loss 0.0823
    epoch 10/10 loss 0.0573
  [inception_v3+bal seed 123] acc 0.9944 | macroF1 0.9935 | SevereRec 0.977 | ProlifRec 1.000
  saved inception_v3_bal_seed123.pth + _preds.npz
Inception V3+bal seed 2025
    epoch 1/10 loss 0.4532
    epoch 2/10 loss 0.1850
    epoch 3/10 loss 0.1265
    epoch 4/10 loss 0.1129
    epoch 5

### Vanilla CNN × 5 (weighted loss)

In [8]:
class VanillaCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,3,padding=1); self.conv2=nn.Conv2d(32,64,3,padding=1)
        self.conv3=nn.Conv2d(64,128,3,padding=1); self.conv4=nn.Conv2d(128,256,3,padding=1)
        self.relu=nn.ReLU(); self.maxPool=nn.MaxPool2d(2)
        self.dropout=nn.Dropout(0.25); self.flatten=nn.Flatten()
        self.fc1=nn.Linear(256*14*14,256); self.fc2=nn.Linear(256,num_classes)
    def forward(self,x):
        x=self.relu(self.maxPool(self.conv1(x))); x=self.relu(self.maxPool(self.conv2(x)))
        x=self.relu(self.maxPool(self.conv3(x))); x=self.relu(self.maxPool(self.conv4(x)))
        x=self.flatten(x); x=self.fc1(x); x=self.dropout(x); return self.fc2(x)

for s in SEEDS:
    if already_done("vanilla_cnn", s): continue
    print(f"Vanilla CNN+bal seed {s}"); set_seed(s); tr, te = make_loaders(224, s)
    m = VanillaCNN().to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    m = train_model(m, tr, crit, opt)
    probs, labels = evaluate(m, te); save_and_report("vanilla_cnn", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

Vanilla CNN+bal seed 42
    epoch 1/10 loss 0.9996
    epoch 2/10 loss 0.6888
    epoch 3/10 loss 0.5423
    epoch 4/10 loss 0.4781
    epoch 5/10 loss 0.3807
    epoch 6/10 loss 0.3454
    epoch 7/10 loss 0.3780
    epoch 8/10 loss 0.3190
    epoch 9/10 loss 0.2753
    epoch 10/10 loss 0.2730
  [vanilla_cnn+bal seed 42] acc 0.6754 | macroF1 0.6261 | SevereRec 0.253 | ProlifRec 0.354
  saved vanilla_cnn_bal_seed42.pth + _preds.npz
Vanilla CNN+bal seed 123
    epoch 1/10 loss 1.0726
    epoch 2/10 loss 0.7280
    epoch 3/10 loss 0.5988
    epoch 4/10 loss 0.4927
    epoch 5/10 loss 0.4432
    epoch 6/10 loss 0.3595
    epoch 7/10 loss 0.3677
    epoch 8/10 loss 0.2966
    epoch 9/10 loss 0.2848
    epoch 10/10 loss 0.3011
  [vanilla_cnn+bal seed 123] acc 0.6959 | macroF1 0.5910 | SevereRec 0.011 | ProlifRec 0.579
  saved vanilla_cnn_bal_seed123.pth + _preds.npz
Vanilla CNN+bal seed 2025
    epoch 1/10 loss 0.9308
    epoch 2/10 loss 0.6833
    epoch 3/10 loss 0.6100
    epoch 4/10 loss 

In [9]:
print("04a done. Save Version (Commit). Then 04b (Swin) and 04b2 (DeiT).")

04a done. Save Version (Commit). Then 04b (Swin) and 04b2 (DeiT).
